# 02 - Fetch a Public Visium Dataset

## Learning objectives
1. Programmatically obtain a public Visium dataset (no credentials).
2. Understand *what files* a Visium dataset is made of, even when a one-liner hides them.
3. Create a clean project directory layout and verify what was fetched.
4. Know the raw 10x download path as a fallback.

## Concept
We use **Squidpy's bundled 10x mouse-brain Visium H&E** sample. The single call
`sq.datasets.visium_hne_adata()` downloads and caches everything and returns a ready
`AnnData` with the image attached. We then save a local copy so later notebooks load fast.


In [1]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


Project root: /Users/justin/Documents/SpatialTranscriptomics-Tutorial


In [2]:
import squidpy as sq
import scanpy as sc

# Folders for raw + processed data (created by the helper; pathlib, no hard-coded paths).
raw = st.raw_dir()
processed = st.processed_dir()
print('raw       ->', raw)
print('processed ->', processed)


raw       -> /Users/justin/Documents/SpatialTranscriptomics-Tutorial/data/raw
processed -> /Users/justin/Documents/SpatialTranscriptomics-Tutorial/data/processed


### Download (cached after the first run)
The first call reaches out to the network and may take a minute; subsequent calls reuse
squidpy's cache. If it fails, re-run the cell and check connectivity (see README
troubleshooting).

In [3]:
adata = st.load_dataset('visium_hne')  # wraps sq.datasets.visium_hne_adata()
adata


AnnData object with n_obs × n_vars = 2688 × 18078
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'leiden', 'cluster'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_colors', 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

**Expected output:** an `AnnData` summary with roughly `n_obs ~ 2,600-2,700` spots and
`n_vars ~ 18,000` genes, plus `obs`, `var`, `uns['spatial']`, and `obsm['spatial']`
populated. (Exact numbers depend on the squidpy version.)


### What is actually inside? (the hidden files)
A raw 10x Visium output folder contains, conceptually:

- **Expression matrix** - `filtered_feature_bc_matrix.h5` (or an `mtx` triplet:
  `matrix.mtx`, `features.tsv`, `barcodes.tsv`) -> becomes `adata.X` + `adata.var` + `adata.obs`.
- **Spatial metadata** - `spatial/tissue_positions(_list).csv` (per-spot pixel coords,
  in/out of tissue) -> becomes `adata.obsm['spatial']` and columns in `adata.obs`.
- **H&E image** - `spatial/tissue_hires_image.png` (+ `tissue_lowres_image.png`) ->
  becomes `adata.uns['spatial'][lib]['images']`.
- **Scale factors** - `spatial/scalefactors_json.json` -> becomes
  `adata.uns['spatial'][lib]['scalefactors']`.

`sq.datasets.visium_hne_adata()` simply parses all of these for you and assembles the
AnnData. Let's confirm each piece is present.


In [4]:
lib = st.get_library_id(adata)
print('Library id:', lib)

spatial_block = adata.uns['spatial'][lib]
print('uns[spatial] keys :', list(spatial_block.keys()))
print('image resolutions :', list(spatial_block['images'].keys()))
print('scalefactor keys  :', list(spatial_block['scalefactors'].keys()))
print('obsm keys         :', list(adata.obsm.keys()))
print('coords shape      :', adata.obsm['spatial'].shape)


Library id: V1_Adult_Mouse_Brain
uns[spatial] keys : ['images', 'metadata', 'scalefactors']
image resolutions : ['hires', 'lowres']
scalefactor keys  : ['fiducial_diameter_fullres', 'spot_diameter_fullres', 'tissue_hires_scalef', 'tissue_lowres_scalef']
obsm keys         : ['X_pca', 'X_umap', 'spatial']
coords shape      : (2688, 2)


**Expected output:** the library id (a string like `'V1_Adult_Mouse_Brain'`), image
resolutions `['hires', 'lowres']`, scale-factor keys including `tissue_hires_scalef`,
`tissue_lowres_scalef`, `spot_diameter_fullres`, and a coords array of shape
`(n_spots, 2)`.


### Save a local copy and print the directory tree
We cache the raw AnnData under `data/processed/` so the next notebooks load instantly,
then print the project's data tree to verify files exist.

In [5]:
raw_h5ad = st.save_adata(adata, 'adata_raw.h5ad')
print('Saved:', raw_h5ad, '\n')

def print_tree(base: Path, prefix: str = ''):
    """Tiny recursive directory printer (depth-limited via recursion on dirs)."""
    entries = sorted(base.iterdir(), key=lambda p: (p.is_file(), p.name))
    for entry in entries:
        size = f'  ({entry.stat().st_size/1e6:.1f} MB)' if entry.is_file() else ''
        print(f'{prefix}{entry.name}{size}')
        if entry.is_dir():
            print_tree(entry, prefix + '    ')

print('data/')
print_tree(st.data_dir(), '    ')


Saved: /Users/justin/Documents/SpatialTranscriptomics-Tutorial/data/processed/adata_raw.h5ad 

data/
    anndata
        visium_hne_adata.h5ad  (329.3 MB)
    processed
        adata_clustered.h5ad  (463.5 MB)
        adata_features.h5ad  (464.0 MB)
        adata_qc.h5ad  (460.9 MB)
        adata_raw.h5ad  (329.3 MB)
    raw
    .gitkeep  (0.0 MB)


**Expected output:** a small tree showing `data/raw/` and `data/processed/adata_raw.h5ad`
(tens to a few hundred MB).


In [6]:
# Verify the cache is loadable (file-existence + integrity check).
assert (st.processed_dir() / 'adata_raw.h5ad').exists(), 'Raw cache missing!'
_check = st.load_adata('adata_raw.h5ad')
print('Reloaded OK:', _check.shape)


Reloaded OK: (2688, 18078)


## Alternative: raw 10x download (fallback)
If you want a *different* dataset (e.g. the clinically relevant **human breast cancer**
section) and you have a stable connection, scanpy can fetch a 10x sample directly. The
code below is intentionally **not executed** here (it is in a markdown block) so this
notebook stays fast and offline-friendly; copy it into a cell to use it.

```python
import scanpy as sc
# Downloads into ./data/raw/<sample>/ a standard 10x 'spatial' folder + matrix.
adata_bc = sc.datasets.visium_sge(
    sample_id='V1_Breast_Cancer_Block_A_Section_1',
)
adata_bc.var_names_make_unique()
```

If a public URL is unstable or rate-limited, **fall back to the squidpy dataset** used
above - the rest of the tutorial is written to work with either (gene panels are filtered
by existence, so species/gene-name differences are handled gracefully).


## Common pitfalls
- **No internet on first run** -> the download fails. Re-run once connected; it caches.
- **Duplicate gene names** -> call `adata.var_names_make_unique()` (squidpy's loader
  already returns unique names; the raw 10x path may not).
- Assuming the data lives in your repo - squidpy caches it in its own directory; we copy a
  working snapshot into `data/processed/`.

## Interpretation
You now have a real Visium dataset on disk and have seen that the convenient one-liner is
just packaging the same files a raw 10x run produces.

## What this means biologically
This is one mouse-brain section: ~2,700 spots tiling the tissue, each with thousands of
gene counts, all registered to an H&E image we will exploit later.

---
**Next:** `03_load_expression_and_spatial_metadata.ipynb` - dissect the AnnData.
